# 05. 통합 평가

세 가지 청킹 전략의 검색 성능 종합 평가

## 1. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import json
from pathlib import Path
import pandas as pd
import numpy as np

from src.evaluators import RetrievalEvaluator
from src.utils.document_loader import DocumentLoader

## 2. 데이터 로드

In [ ]:
# 테스트 쿼리 로드
loader = DocumentLoader()
test_queries = loader.load_test_queries('../data/evaluation/test_queries.json')

print(f"총 {len(test_queries)}개 테스트 쿼리 로드됨")

In [ ]:
# 각 전략의 검색 결과 로드
strategies = ['fixed', 'recursive', 'semantic']
retrieval_results = {}
chunk_stats = {}

for strategy in strategies:
    results_path = f'../results/{strategy}_chunking/retrieval_results.json'
    stats_path = f'../results/{strategy}_chunking/stats.json'
    
    try:
        with open(results_path, 'r') as f:
            retrieval_results[strategy] = json.load(f)
        with open(stats_path, 'r') as f:
            chunk_stats[strategy] = json.load(f)
        print(f"{strategy}: 로드 완료")
    except FileNotFoundError:
        print(f"{strategy}: 파일 없음 - 해당 노트북을 먼저 실행하세요")

## 3. 평가 실행

**참고**: 현재 테스트 쿼리에 Ground Truth가 비어있으므로, 검색 성능은 휴리스틱 기반으로 평가합니다.

In [ ]:
# 평가기 초기화
evaluator = RetrievalEvaluator(k_values=[1, 3, 5, 10])

In [ ]:
# 각 전략별 평가
all_results = {}

for strategy in strategies:
    if strategy not in retrieval_results:
        continue
    
    print(f"\n{'='*60}")
    print(f"{strategy.upper()} 전략 평가")
    print(f"{'='*60}")
    
    # Ground Truth가 비어있으므로 휴리스틱 평가
    # 실제 프로젝트에서는 수동으로 Ground Truth를 작성해야 합니다
    
    # 검색된 청크 분석
    total_retrieved = sum(len(v) for v in retrieval_results[strategy].values())
    queries_with_results = sum(1 for v in retrieval_results[strategy].values() if v)
    
    print(f"총 검색된 청크 수: {total_retrieved}")
    print(f"결과 있는 쿼리 수: {queries_with_results}/{len(test_queries)}")
    
    # 청크 통계
    stats = chunk_stats.get(strategy, {})
    print(f"\n청크 통계:")
    print(f"  - 총 청크 수: {stats.get('total_chunks', 'N/A')}")
    print(f"  - 평균 크기: {stats.get('avg_size', 0):.1f} words")
    print(f"  - 처리 시간: {stats.get('chunking_time', 0):.2f}초")
    
    all_results[strategy] = {
        'total_retrieved': total_retrieved,
        'queries_with_results': queries_with_results,
        'chunk_stats': stats
    }

## 4. 청크 품질 비교

In [ ]:
# 청크 통계 비교 테이블
comparison_data = []

for strategy in strategies:
    stats = chunk_stats.get(strategy, {})
    comparison_data.append({
        'Strategy': strategy.capitalize(),
        'Total Chunks': stats.get('total_chunks', 0),
        'Avg Size': round(stats.get('avg_size', 0), 1),
        'Std Size': round(stats.get('std_size', 0), 1),
        'Min Size': stats.get('min_size', 0),
        'Max Size': stats.get('max_size', 0),
        'Uniformity': round(stats.get('size_uniformity', 0), 3),
        'Time (s)': round(stats.get('chunking_time', 0), 2)
    })

df = pd.DataFrame(comparison_data)
print("\n청크 품질 비교:")
print(df.to_string(index=False))

## 5. 시각화

In [ ]:
import matplotlib.pyplot as plt

# 청크 크기 분포 비교
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['steelblue', 'darkorange', 'forestgreen']

for idx, strategy in enumerate(strategies):
    chunks_path = f'../results/{strategy}_chunking/chunks.json'
    try:
        with open(chunks_path, 'r') as f:
            chunks = json.load(f)
        sizes = [c.get('word_count', len(c.get('text', '').split())) for c in chunks]
        
        ax = axes[idx]
        ax.hist(sizes, bins=25, color=colors[idx], edgecolor='black', alpha=0.7)
        ax.axvline(np.mean(sizes), color='red', linestyle='--', label=f'Mean: {np.mean(sizes):.0f}')
        ax.set_xlabel('Chunk Size (words)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{strategy.capitalize()} (n={len(chunks)})')
        ax.legend()
    except FileNotFoundError:
        axes[idx].text(0.5, 0.5, 'No Data', ha='center', va='center')
        axes[idx].set_title(strategy.capitalize())

plt.suptitle('Chunk Size Distribution by Strategy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/comparison/chunk_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 처리 시간 비교
plt.figure(figsize=(8, 5))

times = [chunk_stats.get(s, {}).get('chunking_time', 0) for s in strategies]
bars = plt.bar([s.capitalize() for s in strategies], times, color=colors)

for bar, t in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{t:.2f}s', ha='center', fontsize=11, fontweight='bold')

plt.ylabel('Processing Time (seconds)')
plt.title('Chunking Processing Time Comparison')
plt.tight_layout()
plt.savefig('../results/comparison/latency_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 결과 저장

In [ ]:
# 비교 결과 저장
comparison_dir = Path('../results/comparison')
comparison_dir.mkdir(parents=True, exist_ok=True)

# JSON 저장
with open(comparison_dir / 'evaluation_results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

# CSV 저장
df.to_csv(comparison_dir / 'chunk_comparison.csv', index=False)

print(f"결과 저장 완료: {comparison_dir}")

## 7. 다음 단계

통합 평가가 완료되었습니다.

다음 노트북: **06_analysis.ipynb** (결과 분석 및 시각화)